# Notebook 1b — Evaluate Θ_o: Output / Linear Probe / NCC

**Purpose:** Re-run only the evaluation section of NB1 on an **existing** checkpoint.  
No training happens here — just load `theta_o_seed{seed}.pt` and compute all 3 metrics.

**Paper:** *An Illusion of Unlearning?* (Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv:2604.08271v1)

**Metrics computed (paper Table 1 — Original row):**
| Metric | Protocol |
|--------|----------|
| Output | Full model forward pass → retain / forget accuracy on test set |
| Linear Probe | Freeze encoder → train `nn.Linear(D,K)` for 50 epochs on full train set → test retain/forget acc |
| NCC | Freeze encoder → class means μ_k from **train** → argmin₂ distance on **test** (paper Eq. 5) |

**Required inputs (set in the Config cell):**
- `CKPT_ROOT` — folder containing `pre_train/theta_o_seed{seed}.pt` and `splits/` JSON files
- `SEEDS`, `RATIOS` — must match the values used when the checkpoints were created

> Set `TEST_MODE = True` for a quick CPU smoke test (2 LP epochs, 1 seed, 1 ratio).

## A. Environment

In [ ]:
import subprocess, sys
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-3000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-1000:])
sh('pip install -q timm einops scikit-learn')

In [ ]:
import os, sys, json, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU    :', torch.cuda.get_device_name(0))

## B. Config — **edit these paths to match your checkpoint**

In [ ]:
# ─── EDIT: set your Kaggle dataset slug ────────────────────────────────────────────
# This is the mount path of the dataset you published from NB1's output.
# On Kaggle it is /kaggle/input/datasets/<username>/<dataset-name>
CKPT_DATASET_DIR = '/kaggle/input/datasets/btk23021592/cmf-notebook1'  # ← EDIT if your slug differs

# Repo dir (needed for model definition import)
REPO_DIR = '/kaggle/working/CMF_Unlearning'

# ── Auto-resolve CKPT_ROOT from the dataset mount (same logic as NB2–5) ─────────────────
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
    # fallback: still in the same session as NB1 (working dir)
    '/kaggle/working/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
]
CKPT_ROOT = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        CKPT_ROOT = os.path.dirname(_p)
        print('Config found:', _p)
        break
assert CKPT_ROOT is not None, (
    'Cannot find cmf_benchmark_config.json.\n'
    'Check that CKPT_DATASET_DIR points to the dataset published from NB1 output.\n'
    'Tried:\n' + '\n'.join(_CONFIG_CANDIDATES)
)

# ── Load seeds/ratios/arch from the saved config (no hard-coding) ────────────────────────
with open(os.path.join(CKPT_ROOT, 'cmf_benchmark_config.json')) as _f:
    _NB1_CFG = __import__('json').load(_f)
SEEDS       = _NB1_CFG['seeds']
RATIOS      = _NB1_CFG['ratios']
DATASET     = _NB1_CFG['dataset']
NUM_CLASSES = _NB1_CFG['num_classes']
ARCH        = _NB1_CFG['arch']
TEST_MODE   = _NB1_CFG.get('test_mode', False)

# Linear Probe — paper config.py EVAL dict: lp_epochs=50, lp_lr=1e-2
LP_MAX_EPOCHS  = 2   if TEST_MODE else 50
LP_LR          = 1e-2
LP_BATCH_SIZE  = 256
NCC_BATCH_SIZE = 256

# Optional override: set True here to force a fast CPU smoke test regardless of NB1 flag
# TEST_MODE = True
if TEST_MODE:
    SEEDS         = SEEDS[:1]
    RATIOS        = RATIOS[:1]
    LP_MAX_EPOCHS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CKPT_ROOT      = {CKPT_ROOT}')
print(f'DATASET={DATASET}  ARCH={ARCH}  NUM_CLASSES={NUM_CLASSES}')
print(f'SEEDS={SEEDS}  RATIOS={RATIOS}  LP_MAX_EPOCHS={LP_MAX_EPOCHS}  device={device}')

## C. Setup — repo import + data

In [ ]:
import subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}',
                   shell=True, check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torchvision, torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                          download=True, transform=transform_train)
test_set   = torchvision.datasets.CIFAR10('/kaggle/working/data', train=False,
                                          download=True, transform=transform_test)

full_train_loader = torch.utils.data.DataLoader(
    full_train, batch_size=128, shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(
    test_set, batch_size=256, shuffle=False, num_workers=2)

print(f'Train: {len(full_train)}  Test: {len(test_set)}')

## D. Load Checkpoints + Splits

In [ ]:
from models.resnet import ResNet18

def build_model():
    return ResNet18(num_classes=NUM_CLASSES, dataset=DATASET).to(device)

# ── load theta_o checkpoints ───────────────────────────────────────────────────
theta_o_models = {}
for seed in SEEDS:
    tag       = f'theta_o_seed{seed}'
    ckpt_path = f'{CKPT_ROOT}/pre_train/{tag}.pt'
    assert os.path.exists(ckpt_path), f'Checkpoint not found: {ckpt_path}'
    ck    = torch.load(ckpt_path, map_location=device)
    # support both raw state-dict and wrapped {'model_state_dict': ...}
    state = ck['model_state_dict'] if isinstance(ck, dict) and 'model_state_dict' in ck else ck
    model = build_model()
    model.load_state_dict(state)
    model.eval()
    theta_o_models[seed] = model
    print(f'[seed{seed}] loaded {ckpt_path}')

# ── load forget / retain split indices ──────────────────────────────────────────
split_info = {}
for ratio in RATIOS:
    for seed in SEEDS:
        tag   = f'ratio{ratio}_seed{seed}'
        fpath = f'{CKPT_ROOT}/splits/forget_indices_{tag}.json'
        rpath = f'{CKPT_ROOT}/splits/retain_indices_{tag}.json'
        assert os.path.exists(fpath), f'Split file not found: {fpath}'
        with open(fpath) as f: forget_idx = json.load(f)
        with open(rpath) as f: retain_idx = json.load(f)
        split_info[(ratio, seed)] = {'forget': forget_idx, 'retain': retain_idx}
        print(f'[{tag}] forget={len(forget_idx)}  retain={len(retain_idx)}')

print('\nAll checkpoints and splits loaded.')

## E. Evaluation Helpers

In [ ]:
# ── helper 1: output accuracy ────────────────────────────────────────────────────────
@torch.no_grad()
def output_retain_forget(model, test_set, forget_indices):
    """Top-1 accuracy on the retain and forget subsets of test_set."""
    train_targets  = np.array(full_train.targets)
    forget_classes = sorted(set(train_targets[forget_indices].tolist()))
    model.eval()
    all_pred, all_true = [], []
    loader = torch.utils.data.DataLoader(
        test_set, batch_size=256, shuffle=False, num_workers=2)
    for x, y in loader:
        all_pred.extend(model(x.to(device)).argmax(1).cpu().tolist())
        all_true.extend(y.tolist())
    pred   = np.array(all_pred);  true = np.array(all_true)
    f_mask = np.isin(true, forget_classes);  r_mask = ~f_mask
    ret_acc = 100.0 * (pred[r_mask] == true[r_mask]).mean() if r_mask.any() else float('nan')
    for_acc = 100.0 * (pred[f_mask] == true[f_mask]).mean() if f_mask.any() else float('nan')
    return ret_acc, for_acc, forget_classes


# ── helper 2: penultimate-layer feature extraction ─────────────────────────────
@torch.no_grad()
def extract_features(model, loader):
    """Hook onto model.avgpool, return [N, D] float32 features and [N] labels."""
    feats, labels, buf = [], [], []
    def _hook(_m, _inp, out):
        buf.append(out.view(out.size(0), -1).detach().cpu())
    handle = model.avgpool.register_forward_hook(_hook)
    model.eval()
    for x, y in loader:
        buf.clear()
        _ = model(x.to(device))
        feats.append(buf[0]);  labels.append(y)
    handle.remove()
    return torch.cat(feats, 0).float(), torch.cat(labels, 0).long()


# ── helper 3: linear probe ─────────────────────────────────────────────────────────────────
def linear_probe_eval(model, train_loader_full, test_set, forget_classes):
    """Train nn.Linear on frozen features for LP_MAX_EPOCHS epochs.

    Paper Sec 3.2: LP trained on full D = D_r u D_f.
    """
    probe = copy.deepcopy(model).to(device)
    probe.eval()
    for p in probe.parameters(): p.requires_grad_(False)

    Xtr, ytr = extract_features(probe, train_loader_full)
    Xte, yte = extract_features(probe, torch.utils.data.DataLoader(
        test_set, batch_size=LP_BATCH_SIZE, shuffle=False, num_workers=2))

    clf = nn.Linear(Xtr.size(1), NUM_CLASSES).to(device)
    opt = optim.SGD(clf.parameters(), lr=LP_LR, momentum=0.9, weight_decay=0.0)
    dl  = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr),
        batch_size=LP_BATCH_SIZE, shuffle=True)

    for ep in range(LP_MAX_EPOCHS):
        clf.train()
        for bx, by in dl:
            opt.zero_grad()
            nn.CrossEntropyLoss()(clf(bx.to(device)), by.to(device)).backward()
            opt.step()
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f'  LP epoch {ep+1}/{LP_MAX_EPOCHS}')

    clf.eval()
    with torch.no_grad():
        pred_te = clf(Xte.to(device)).argmax(1).cpu().numpy()
    true_te = yte.numpy()
    f_mask  = np.isin(true_te, forget_classes);  r_mask = ~f_mask
    ret = 100.0 * (pred_te[r_mask] == true_te[r_mask]).mean() if r_mask.any() else float('nan')
    frg = 100.0 * (pred_te[f_mask] == true_te[f_mask]).mean() if f_mask.any() else float('nan')
    return ret, frg


# ── helper 4: NCC (paper Eq. 5) ─────────────────────────────────────────────────────────────
def ncc_eval(model, train_loader_full, test_set, forget_classes):
    """NCC accuracy: class means from TRAIN, argmin-distance on TEST (paper Eq. 5).

    Protocol:
      mu_k = mean of avgpool features for class k over the full TRAIN set.
      pred = argmin_k ||phi(x) - mu_k||_2  on each TEST sample.
    """
    probe = copy.deepcopy(model).to(device)
    probe.eval()
    for p in probe.parameters(): p.requires_grad_(False)

    # step 1: compute class means from train features
    Xtr, ytr = extract_features(probe, train_loader_full)
    means = []
    for c in range(NUM_CLASSES):
        mask = (ytr == c)
        means.append(Xtr[mask].mean(0) if mask.any() else torch.zeros(Xtr.size(1)))
    M = torch.stack(means).to(device)          # [K, D]

    # step 2: classify test samples by nearest class centre
    Xte, yte = extract_features(probe, torch.utils.data.DataLoader(
        test_set, batch_size=NCC_BATCH_SIZE, shuffle=False, num_workers=2))
    dists = torch.cdist(Xte.to(device).unsqueeze(0),
                        M.unsqueeze(0)).squeeze(0)  # [N, K]
    pred  = dists.argmin(1).cpu().numpy()
    true  = yte.numpy()

    # step 3: retain / forget split
    f_mask = np.isin(true, forget_classes);  r_mask = ~f_mask
    ret = 100.0 * (pred[r_mask] == true[r_mask]).mean() if r_mask.any() else float('nan')
    frg = 100.0 * (pred[f_mask] == true[f_mask]).mean() if f_mask.any() else float('nan')
    return ret, frg


print('Helpers defined: output_retain_forget | extract_features | linear_probe_eval | ncc_eval')

## F. Run Evaluation (all seeds × ratios)

In [ ]:
eval_rows = []

for seed in SEEDS:
    model = theta_o_models[seed]
    for ratio in RATIOS:
        forget_idx     = split_info[(ratio, seed)]['forget']
        train_targets  = np.array(full_train.targets)
        forget_classes = sorted(set(train_targets[forget_idx].tolist()))

        print(f'\n[seed={seed} ratio={ratio}]  forget_classes={forget_classes}')

        # 1) Output accuracy
        out_ret, out_for, _ = output_retain_forget(model, test_set, forget_idx)
        print(f'  Output   retain={out_ret:.2f}%  forget={out_for:.2f}%')

        # 2) Linear Probe (paper Sec 3.2: full train set, 50 epochs)
        lp_ret, lp_for = linear_probe_eval(model, full_train_loader, test_set, forget_classes)
        print(f'  LP       retain={lp_ret:.2f}%  forget={lp_for:.2f}%')

        # 3) NCC (paper Eq. 5: train centres -> test evaluation)
        ncc_ret, ncc_for = ncc_eval(model, full_train_loader, test_set, forget_classes)
        print(f'  NCC      retain={ncc_ret:.2f}%  forget={ncc_for:.2f}%')

        eval_rows.append({
            'seed': seed, 'ratio': ratio,
            'forget_classes': str(forget_classes),
            'out_retain':  round(out_ret,  2),  'out_forget':  round(out_for,  2),
            'lp_retain':   round(lp_ret,   2),  'lp_forget':   round(lp_for,   2),
            'ncc_retain':  round(ncc_ret,  2),  'ncc_forget':  round(ncc_for,  2),
        })

df_eval = pd.DataFrame(eval_rows)
print('\n=== Theta_o Evaluation (all seeds x ratios) ===')
print(df_eval[['seed', 'ratio',
               'out_retain',  'out_forget',
               'lp_retain',   'lp_forget',
               'ncc_retain',  'ncc_forget']].to_string(index=False))

## G. Save Results

In [ ]:
eval_path = f'{CKPT_ROOT}/theta_o_eval.json'
with open(eval_path, 'w') as f:
    json.dump(eval_rows, f, indent=2)
print(f'Saved: {eval_path}')

print()
print('=== Summary ===')
print(df_eval[['seed', 'ratio',
               'out_retain',  'out_forget',
               'lp_retain',   'lp_forget',
               'ncc_retain',  'ncc_forget']].to_string(index=False))
print()
print('Output    = full model forward pass (paper Table 1 Output row)')
print('LP        = linear probe on frozen features (paper Table 1 Linear Probe row)')
print('NCC       = nearest-class-center on frozen features (paper Table 1 NCC row)')